# Install all required dependencies

In [1]:
! python -m pip install librosa soundfile numpy pandas matplotlib seaborn tqdm requests

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------- -------------------- 1.3/2.8 MB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 6.7 MB/s  0:00:00
   ---------------------------------------- 0.0/39.2 MB ? eta -:--:--
   -- ------------------------------------- 2.1/39.2 MB 11.5 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/39.2 MB 12.0 MB/s eta 0:00:03
   ------- -------------------------------- 7.1/39.2 MB 11.8 MB/s eta 0:00:03
   --------- ------------------------------ 9.7/39.2 MB 12.0 MB/s eta 0:00:03
   ------------ --------------------------- 12.3/39.2 MB 11.9 MB/s eta 0:00:03
   --------------- ------------------------ 14.9/39.2 MB 12.1 MB/s eta 0:00:03
   ------------------ --------------------- 17.8/39.2 MB 12.0 MB/s eta 0:00:02
   ------------------

# Creating Project Folder Structure

In [5]:
import os

langs = ['malayalam', 'tamil', 'hindi', 'english', 'kannada']
folders = ['data/raw', 'data/processed', 'data/metadata', 'data/splits', 'downloads']
for lang in langs:
    folders.append(f'data/raw/{lang}')
    folders.append(f'data/processed/{lang}')

for f in folders:
    os.makedirs(f, exist_ok=True)
print("All folders created.")

All folders created.


# Defining OpenSLR sources

In [6]:
#  filenames from openslr.org pages
# Using EU mirror (openslr.elda.org) — more reliable than main server

BASE = 'https://openslr.elda.org/resources'

openslr_sources = {
    'malayalam': f'{BASE}/63/ml_in_female.zip',   # SLR63 ~710MB
    'tamil':     f'{BASE}/65/ta_in_female.zip',   # SLR65 ~769MB
    'kannada':   f'{BASE}/79/kn_in_female.zip',   # SLR79 ~500MB
    'hindi':     f'{BASE}/103/Hindi_test.tar.gz', # SLR103 ~258MB (test split, smaller)
    'english':   f'{BASE}/45/ST-AEDS-20180100_1-OS.tgz', # SLR45 ~350MB free American English
}

print("Sources:")
for lang, url in openslr_sources.items():
    print(f"  {lang}: {url}")

Sources:
  malayalam: https://openslr.elda.org/resources/63/ml_in_female.zip
  tamil: https://openslr.elda.org/resources/65/ta_in_female.zip
  kannada: https://openslr.elda.org/resources/79/kn_in_female.zip
  hindi: https://openslr.elda.org/resources/103/Hindi_test.tar.gz
  english: https://openslr.elda.org/resources/45/ST-AEDS-20180100_1-OS.tgz


# Download all datasets with progress bar

In [7]:
import urllib.request
from tqdm import tqdm
import os

def download_file(url, dest_path):
    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / 1e6
        print(f"  Already exists ({size_mb:.1f} MB): {os.path.basename(dest_path)}")
        return True
    
    print(f"  Downloading: {os.path.basename(dest_path)}")
    print(f"  From: {url}")
    
    class ProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)
    
    try:
        with ProgressBar(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(dest_path)) as t:
            urllib.request.urlretrieve(url, dest_path, reporthook=t.update_to)
        size_mb = os.path.getsize(dest_path) / 1e6
        print(f"  Saved ({size_mb:.1f} MB): {dest_path}")
        return True
    except Exception as e:
        print(f"  ERROR: {e}")
        if os.path.exists(dest_path):
            os.remove(dest_path)  # remove partial download
        return False

failed = []
for lang, url in openslr_sources.items():
    print(f"\n--- {lang.upper()} ---")
    fname = url.split('/')[-1]
    dest = f'downloads/{lang}_{fname}'
    success = download_file(url, dest)
    if not success:
        failed.append(lang)

if failed:
    print(f"\nFailed downloads: {failed}")
    print("Try the CN mirror: replace 'openslr.elda.org' with 'openslr.magicdatatech.com'")
else:
    print("\nAll downloads complete.")


--- MALAYALAM ---
  Downloading: malayalam_ml_in_female.zip
  From: https://openslr.elda.org/resources/63/ml_in_female.zip


malayalam_ml_in_female.zip: 710MB [01:16, 9.33MB/s]                               


  Saved (710.2 MB): downloads/malayalam_ml_in_female.zip

--- TAMIL ---
  Downloading: tamil_ta_in_female.zip
  From: https://openslr.elda.org/resources/65/ta_in_female.zip


tamil_ta_in_female.zip: 770MB [01:24, 9.09MB/s]                               


  Saved (769.5 MB): downloads/tamil_ta_in_female.zip

--- KANNADA ---
  Downloading: kannada_kn_in_female.zip
  From: https://openslr.elda.org/resources/79/kn_in_female.zip


...

  Saved (980.8 MB): downloads/kannada_kn_in_female.zip

--- HINDI ---
  Downloading: hindi_Hindi_test.tar.gz
  From: https://openslr.elda.org/resources/103/Hindi_test.tar.gz


hindi_Hindi_test.tar.gz: 259MB [00:38, 6.71MB/s]                              


  Saved (259.0 MB): downloads/hindi_Hindi_test.tar.gz

--- ENGLISH ---
  Downloading: english_ST-AEDS-20180100_1-OS.tgz
  From: https://openslr.elda.org/resources/45/ST-AEDS-20180100_1-OS.tgz


english_ST-AEDS-20180100_1-OS.tgz: 351MB [00:37, 9.48MB/s]                              

  Saved (351.4 MB): downloads/english_ST-AEDS-20180100_1-OS.tgz

All downloads complete.


# Extracting Archives

In [8]:
import zipfile, tarfile, os

def extract_archive(archive_path, extract_to):
    print(f"  Extracting: {os.path.basename(archive_path)} → {extract_to}")
    if archive_path.endswith('.zip'):
        with zipfile.ZipFile(archive_path, 'r') as z:
            z.extractall(extract_to)
    elif archive_path.endswith(('.tar.gz', '.tgz')):
        with tarfile.open(archive_path, 'r:gz') as t:
            t.extractall(extract_to)
    print(f"  Done.")

for lang, url in openslr_sources.items():
    fname = url.split('/')[-1]
    archive_path = f'downloads/{lang}_{fname}'
    if os.path.exists(archive_path):
        extract_archive(archive_path, f'data/raw/{lang}/')
    else:
        print(f"Skipping {lang} — archive not found")

  Extracting: malayalam_ml_in_female.zip → data/raw/malayalam/
  Done.
  Extracting: tamil_ta_in_female.zip → data/raw/tamil/
  Done.
  Extracting: kannada_kn_in_female.zip → data/raw/kannada/
  Done.
  Extracting: hindi_Hindi_test.tar.gz → data/raw/hindi/
  Done.
  Extracting: english_ST-AEDS-20180100_1-OS.tgz → data/raw/english/
  Done.


# Flatten nested folders, collect WAVs

In [9]:
import shutil, os

MAX_PER_LANG = 300

for lang in langs:
    raw_dir = f'data/raw/{lang}'
    collected = 0
    
    for root, dirs, files in os.walk(raw_dir):
        if root == raw_dir:
            continue  # skip top level, only walk subdirs
        for fname in files:
            if fname.lower().endswith(('.wav', '.flac')):
                src = os.path.join(root, fname)
                dest = os.path.join(raw_dir, fname)
                if not os.path.exists(dest):
                    shutil.copy2(src, dest)
                collected += 1
                if collected >= MAX_PER_LANG:
                    break
        if collected >= MAX_PER_LANG:
            break

    total = len([f for f in os.listdir(raw_dir) if f.endswith(('.wav','.flac'))])
    print(f"{lang}: {total} audio files in raw/")

malayalam: 2103 audio files in raw/
tamil: 2335 audio files in raw/
hindi: 300 audio files in raw/
english: 3842 audio files in raw/
kannada: 2186 audio files in raw/


# Convert to 16kHz mono WAV 

In [10]:
import librosa, soundfile as sf, os
from tqdm import tqdm

SAMPLE_RATE = 16000
errors = []

for lang in langs:
    raw_dir = f'data/raw/{lang}'
    out_dir = f'data/processed/{lang}'
    files = [f for f in os.listdir(raw_dir) if f.lower().endswith(('.wav', '.flac'))]
    print(f"\n{lang}: {len(files)} files to convert")
    
    for fname in tqdm(files, desc=lang):
        in_path = os.path.join(raw_dir, fname)
        out_name = os.path.splitext(fname)[0] + '.wav'
        out_path = os.path.join(out_dir, out_name)
        if os.path.exists(out_path):
            continue
        try:
            y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)
            sf.write(out_path, y, SAMPLE_RATE)
        except Exception as e:
            errors.append((lang, fname, str(e)))

print(f"\nDone. Errors: {len(errors)}")
for e in errors[:5]:
    print(f"  {e}")


malayalam: 2103 files to convert


malayalam:   0%|          | 0/2103 [00:00<?, ?it/s]c:\Users\Shipra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
malayalam: 100%|██████████| 2103/2103 [01:19<00:00, 26.30it/s] 



tamil: 2335 files to convert


tamil: 100%|██████████| 2335/2335 [00:52<00:00, 44.74it/s]



hindi: 300 files to convert


...


english: 3842 files to convert


english: 100%|██████████| 3842/3842 [00:46<00:00, 82.64it/s] 



kannada: 2186 files to convert


kannada: 100%|██████████| 2186/2186 [00:46<00:00, 47.04it/s]


Done. Errors: 0


# Filter by duration 3-10s

In [11]:
import librosa, os
from tqdm import tqdm

MIN_DUR, MAX_DUR = 3.0, 10.0
kept = removed = 0

for lang in langs:
    proc_dir = f'data/processed/{lang}'
    for fname in tqdm(os.listdir(proc_dir), desc=lang):
        if not fname.endswith('.wav'):
            continue
        fpath = os.path.join(proc_dir, fname)
        try:
            dur = librosa.get_duration(path=fpath)
            if MIN_DUR <= dur <= MAX_DUR:
                kept += 1
            else:
                os.remove(fpath); removed += 1
        except:
            os.remove(fpath); removed += 1

print(f"\nKept: {kept}   Removed: {removed}")

kannada: 100%|██████████| 2186/2186 [00:03<00:00, 704.40it/s]


Kept: 9166   Removed: 1600


# Build Metadata CSV

In [12]:
import pandas as pd, librosa, os
from tqdm import tqdm

records = []
for lang in langs:
    proc_dir = f'data/processed/{lang}'
    for fname in tqdm(sorted(os.listdir(proc_dir)), desc=lang):
        if not fname.endswith('.wav'):
            continue
        fpath = os.path.join(proc_dir, fname)
        try:
            dur = librosa.get_duration(path=fpath)
            records.append({'file_path': fpath, 'file_name': fname,
                            'language': lang, 'duration_sec': round(dur, 3),
                            'sample_rate': 16000, 'source': 'openslr'})
        except:
            pass

df = pd.DataFrame(records)
df.to_csv('data/metadata/metadata.csv', index=False)

print(f"\nTotal: {len(df)} clips")
print(df.groupby('language').agg(
    clips=('file_name','count'),
    avg_sec=('duration_sec','mean'),
    total_min=('duration_sec', lambda x: round(x.sum()/60, 1))
).round(2))

kannada: 100%|██████████| 1617/1617 [00:01<00:00, 1065.88it/s]



Total: 9166 clips
           clips  avg_sec  total_min
language                            
english     3315     4.74      262.0
hindi        298     5.71       28.3
kannada     1617     6.37      171.7
malayalam   1761     5.38      157.8
tamil       2175     6.11      221.5
